# Calculating Indices: SRI, SCAI, and GAI

In this notebook, we will calculate the Stroke Risk Index (SRI), Stroke Care Access Index (SCAI), and Geographic Accessibility Index (GAI), using predefined variables.

## Reading Data from Database and Importing Index Pipeline

In [1]:
# Importing packages

import sqlite3
import pandas as pd
import sys

In [2]:
sys.path.insert(0, "../src")

In [3]:
# Importing pipeline
from index_pipeline import build_index

In [4]:
con = sqlite3.connect("../data/stroke_burden.db")

In [5]:
df = pd.read_sql("SELECT * FROM master", con)

In [6]:
# Creating SCAI dataframe

scai = df[[
    "fips",
    "hospitals_per_100k",
    "hospital_beds_per_100k",
    "pcp_per_100k",
    "neurologists_per_100k",
    "stroke_centers_per_100k",
    "pcnt_insured"
]].copy()

In [7]:
# Creating GAI dataframe

gai = df[[
    "fips",
    "drive_time_min",
    "drive_time_advanced",
    "nearest_stroke_distance",
    "nearest_stroke_distance_advanced"
]].copy()

In [8]:
# Creating SRI dataframe

sri = df[[
    "fips",
    "pop_density",
    "pcnt_65_plus",
    "poverty_rate",
    "pcnt_low_income",
    "pcnt_bachelors",
    "smoking_prevalence",
    "obesity_prevalence",
    "diabetes_prevalence",
    "physical_inactivity",
    "hypertension_prevalence",
    "high_cholesterol_prevalence",
    "binge_drinking_prevalence",
    "stroke_prevalence"
]].copy()

## Building Indices

### SCAI

In [9]:
scai_result = build_index(
    scai,
    [
        "hospitals_per_100k",        
        "hospital_beds_per_100k",
        "pcp_per_100k",
        "neurologists_per_100k",
        "stroke_centers_per_100k",
        "pcnt_insured"
    ]
)

df["scai"] = scai_result.scores

In [10]:
scai_result.explained_variance_ratio

0.3757180827142832

In [11]:
scai_result.loadings

hospitals_per_100k        -0.181756
hospital_beds_per_100k     0.495998
pcp_per_100k               0.584829
neurologists_per_100k      0.574440
stroke_centers_per_100k    0.195230
pcnt_insured               0.104065
Name: index_pc1_loading, dtype: float64

### GAI

In [12]:
gai_result = build_index(
    gai,
    [
        "drive_time_min",
        "drive_time_advanced",
        "nearest_stroke_distance",
        "nearest_stroke_distance_advanced"
    ],
    flip=[
        "drive_time_min",
        "drive_time_advanced",
        "nearest_stroke_distance",
        "nearest_stroke_distance_advanced"
    ],
    transforms={
        "drive_time_min":"log1p",
        "drive_time_advanced":"log1p",
        "nearest_stroke_distance":"log1p",
        "nearest_stroke_distance_advanced":"log1p"
    }
)

df["gai"] = gai_result.scores

In [13]:
gai_result.explained_variance_ratio

0.7706484916219887

In [14]:
gai_result.loadings

drive_time_min                      0.487536
drive_time_advanced                 0.508339
nearest_stroke_distance             0.505386
nearest_stroke_distance_advanced    0.498483
Name: index_pc1_loading, dtype: float64

### SRI

In [15]:
sri_result = build_index(
    sri,
    [
        "pop_density",
        "pcnt_65_plus",
        "poverty_rate",
        "pcnt_low_income",
        "pcnt_bachelors",
        "smoking_prevalence",
        "obesity_prevalence",
        "diabetes_prevalence",
        "physical_inactivity",
        "hypertension_prevalence",
        "high_cholesterol_prevalence",
        "binge_drinking_prevalence",
        "stroke_prevalence"
    ],
    flip=["pcnt_bachelors"],
    transforms={"pop_density": "log1p"},
    name="sri"
)

In [16]:
sri_result.explained_variance_ratio

0.5223875266272118

In [17]:
sri_result.loadings

pop_density                   -0.220684
pcnt_65_plus                   0.155486
poverty_rate                   0.245481
pcnt_low_income                0.325243
pcnt_bachelors                 0.351806
smoking_prevalence             0.361232
obesity_prevalence             0.296924
diabetes_prevalence            0.294431
physical_inactivity            0.297536
hypertension_prevalence        0.324855
high_cholesterol_prevalence    0.064245
binge_drinking_prevalence     -0.004078
stroke_prevalence              0.365741
Name: sri_pc1_loading, dtype: float64